In [ ]:
# NOTE: Notebook này khớp với commit 2ab2bfd của repo AlexKeatg-FS.
# Sau khi clone, cell này sẽ in commit thực tế — nếu không khớp thì chạy: git -C /content/AlexKeatg-FS pull
import os
import shutil
import requests

REPO = 'AlexKeatg-FS'
BASE = '/content'
DEST = f'{BASE}/{REPO}'

# Lấy token: Colab Secrets (userdata) -> env -> nhập tay
TOKEN = None
try:
    from google.colab import userdata
    TOKEN = userdata.get('GITHUB_TOKEN')
except Exception:
    pass
if not TOKEN:
    TOKEN = os.environ.get('GITHUB_TOKEN')
if not TOKEN:
    import getpass
    TOKEN = getpass.getpass('GitHub token (scope repo): ')

headers = {'Authorization': f'token {TOKEN}'}
resp = requests.get('https://api.github.com/user', headers=headers)
assert resp.status_code == 200, f'Token không hợp lệ: {resp.status_code} {resp.text[:200]}'
USER = resp.json()['login']
print('GitHub user:', USER)

if os.path.exists(DEST):
    shutil.rmtree(DEST)
!git clone --depth 1 https://{TOKEN}@github.com/{USER}/{REPO}.git {DEST}
assert os.path.exists(f'{DEST}/run.py'), 'Clone thất bại!'
!git -C {DEST} log --oneline -1
print('✅ Clone OK ->', DEST)

In [ ]:
%cd /content/AlexKeatg-FS
import os, subprocess
from pathlib import Path

MAMBA = Path('/content/micromamba/bin/micromamba')
MAMBA_ROOT = Path('/content/mamba_root')
ENV = MAMBA_ROOT / 'envs' / 'alexkeatg310'
MAIN_PY = ENV / 'bin' / 'python'

def run(cmd):
    cmd = [str(x) for x in cmd]
    print('$', ' '.join(cmd), flush=True)
    p = subprocess.run(cmd, check=False)
    if p.returncode != 0:
        raise RuntimeError('Command failed: ' + ' '.join(cmd))
    return p

# 1) Micromamba (chỉ cài lần đầu)
if not MAMBA.exists():
    run(['bash', '-lc',
         'mkdir -p /content/micromamba && '
         'curl -L --fail --retry 5 -s https://micro.mamba.pm/api/micromamba/linux-64/latest '
         '| tar -xj -C /content/micromamba'])
assert MAMBA.exists(), MAMBA

# 2) Env python 3.10 (chỉ tạo lần đầu)
if not MAIN_PY.exists():
    e = os.environ.copy(); e['MAMBA_ROOT_PREFIX'] = str(MAMBA_ROOT)
    subprocess.run([str(MAMBA), 'create', '-y', '-n', 'alexkeatg310', '-c', 'conda-forge', 'python=3.10', 'pip'], env=e, check=True)

# 3) Cài packages vào ENV RIÊNG -> không đụng kernel Colab -> KHÔNG nhắc restart
# Gỡ torch nếu env cũ còn (đã bỏ hẳn — mọi thứ chạy ONNX)
run([MAIN_PY, '-m', 'pip', 'uninstall', '-y', 'torch', 'torchvision'])
run([MAIN_PY, '-m', 'pip', 'install', '-q', '--upgrade', 'pip', 'setuptools', 'wheel'])
run([MAIN_PY, '-m', 'pip', 'install', '-q', 'numpy<2.0', 'opencv-python-headless', 'onnx', 'insightface',
     'scipy', 'scikit-image', 'psutil', 'tqdm', 'requests', 'fastapi', 'yt-dlp', 'gradio==5.13.0', 'pycryptodomex'])
run([MAIN_PY, '-m', 'pip', 'install', '-q', '--force-reinstall', 'pydantic==2.10.6'])
run([MAIN_PY, '-m', 'pip', 'uninstall', '-y', 'onnxruntime', 'onnxruntime-gpu'])
run([MAIN_PY, '-m', 'pip', 'install', '-q', 'onnxruntime-gpu'])  # bản mới nhất (không cần pin)

# 4) Verify CUDA trong env
run([MAIN_PY, '-c',
     'import onnxruntime as ort; '
     'print("ORT providers:", ort.get_available_providers())'])
print('✅ Env OK — KHÔNG cần restart runtime. Chạy cell 3 (tải video) rồi cell 5 (chạy app).')

In [ ]:
# === Cell 3: Download video (m3u8 / mp4) từ URL ===
# Kiểm tra yt-dlp, cài nếu thiếu
import subprocess, sys
try:
    import yt_dlp
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'yt-dlp'])
    import yt_dlp

import os, time
from pathlib import Path

TARGET_DIR = '/content/target_files'
os.makedirs(TARGET_DIR, exist_ok=True)

def download_video(url, output_dir=TARGET_DIR):
    """Download m3u8 hoặc mp4 từ URL. Hiển thị tiến trình.
    Trả về đường dẫn file đã tải."""
    os.makedirs(output_dir, exist_ok=True)
    url = url.strip()
    if not url:
        print('❌ URL không được để trống!')
        return None

    is_m3u8 = '.m3u8' in url.lower() or 'm3u8' in url.lower()
    fmt = 'M3U8/HLS' if is_m3u8 else 'MP4/Direct'
    print(f'📥 Đang tải [{fmt}]: {url[:120]}')

    last_percent = [-1]
    start_time = [time.time()]

    def progress_hook(d):
        if d['status'] == 'downloading':
            total = d.get('total_bytes') or d.get('total_bytes_estimate') or 0
            downloaded = d.get('downloaded_bytes', 0)
            speed = d.get('speed')
            eta = d.get('eta')
            if total > 0:
                pct = downloaded * 100 // total
                if pct != last_percent[0]:
                    elapsed = time.time() - start_time[0]
                    speed_str = f'{speed/1024/1024:.1f} MB/s' if speed else 'tính...'
                    eta_str = f'{eta}s' if eta else '?'
                    dl_mb = downloaded / 1024 / 1024
                    total_mb = total / 1024 / 1024
                    print(f'   ⏳ {pct}%  ({dl_mb:.1f}/{total_mb:.1f} MB)  Tốc độ: {speed_str}  Còn lại: {eta_str}', flush=True)
                    last_percent[0] = pct
        elif d['status'] == 'finished':
            elapsed = time.time() - start_time[0]
            size_mb = d.get('downloaded_bytes', 0) / 1024 / 1024
            print(f'   ✅ Tải xong! {size_mb:.1f} MB trong {elapsed:.1f}s')

    ydl_opts = {
        'outtmpl': os.path.join(output_dir, '%(title)s.%(ext)s'),
        'progress_hooks': [progress_hook],
        'noprogress': False,
        'quiet': True,
        'no_warnings': True,
    }
    if is_m3u8:
        ydl_opts['hls_prefer_native'] = True
    else:
        # mp4 trực tiếp — dùng wget-like approach qua yt-dlp
        ydl_opts['http_headers'] = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
        }

    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            info = ydl.extract_info(url, download=True)
            filename = ydl.prepare_filename(info)
        if os.path.exists(filename):
            print(f'📁 Lưu tại: {filename}')
            return filename
        # fallback: tìm file mới nhất trong output_dir
        files = sorted(Path(output_dir).iterdir(), key=os.path.getmtime, reverse=True)
        if files:
            f = str(files[0])
            print(f'📁 Lưu tại: {f}')
            return f
        print('❌ Không tìm thấy file sau khi tải!')
        return None
    except Exception as e:
        print(f'❌ Lỗi tải video: {e}')
        return None

# --- Demo: tải 1 URL (thay link bên dưới) ---
# download_video('https://example.com/video.mp4')

print('✅ Hàm download_video sẵn sàng!')
print(f'📂 Thư mục lưu: {TARGET_DIR}')
print('Sử dụng: download_video("https://...")')

In [ ]:
# === Cell 4: Tải video từ URL (chạy cell này mỗi lần cần tải) ===
# Dán URL vào ô dưới rồi chạy cell

VIDEO_URL = ''  # <-- Dán link m3u8 hoặc mp4 vào đây

if VIDEO_URL.strip():
    result = download_video(VIDEO_URL)
    if result:
        print(f'\n🎉 File sẵn sàng! Vào app → "Add local files from" → nhập: {TARGET_DIR}')
else:
    print('⚠️ Chưa nhập URL. Hãy dán link m3u8/mp4 vào biến VIDEO_URL rồi chạy lại cell này.')

In [ ]:
%cd /content/AlexKeatg-FS
!rm -f config.yaml  # luôn dùng defaults mới (threads 16, crf 22, DFL XSeg, erosion 2)
import os
PY = '/content/mamba_root/envs/alexkeatg310/bin'
assert os.path.exists(PY + '/python'), 'Chạy cell 2 trước!'
os.environ['PATH'] = PY + ':' + os.environ.get('PATH', '')
os.environ['MPLBACKEND'] = 'Agg'
os.environ['GRADIO_ANALYTICS_ENABLED'] = 'False'

# Chép Colab Secrets sang env + file để app (subprocess mamba) đọc được
def export_secret(name):
    try:
        from google.colab import userdata
        val = userdata.get(name)
    except Exception:
        val = None
    val = val or os.environ.get(name)
    if val:
        os.environ[name] = val
        try:
            with open('/content/.secret_' + name, 'w') as f:
                f.write(val)
        except Exception:
            pass
    return val
for name in ['mega_email', 'mega_pass', 'tr_pass']:
    if export_secret(name):
        print(f'✅ Secret {name} đã sẵn sàng')

!{PY}/python -u run.py